[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/22_capstone/22_capstone.ipynb)

# Módulo 22 · Capstone: la plataforma completa de observabilidad de SupportOps

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración estimada** | ≈ 4 h 30 min (guía y plataforma de referencia ≈ 2 h 30 min + entrega final ≈ 2 h) |
| **Servicios de Compose** | Opcional: `--profile core --profile metrics` (sección 11, ≈ 4 min: el Collector del capstone se conecta al stack). Docker para validar el Collector y el *Collector real* efímero. Sin Docker, modo degradado |
| **RAM aproximada** | < 700 MB sin stack · ≈ 1,3 GB con `core` + `metrics` + el gateway efímero |
| **Prerrequisitos** | Todo el curso (00-21). **No hay conceptos nuevos**: es integración |
| **Hilo conductor** | El agente de SupportOps empezó siendo un bucle de 80 líneas. Hoy es un **sistema multi-agente observado, con SLOs, calidad, costes, privacidad, muestreo y despliegue**, con un *scorecard* que lo prueba y una revisión de arquitectura que lo defiende |

> ⚠️ **Honestidad sobre el laboratorio.** Todo es simulado y determinista (`mock`, semilla 42): el tráfico (≈ 4 tickets/min), el incidente del proveedor y el análisis de muestreo y coste (corpus del módulo 18, de **agente único**; el multi-agente aporta los contratos, el SLO y la privacidad).
> Los precios son ilustrativos. Kubernetes (módulo 19) no ejecutó pods en este entorno: lo que se verifica del despliegue son los valores y las configuraciones, no un clúster en marcha. Un capstone «de verdad» te pide además probar la plataforma bajo **tu** carga y **tus** datos.

## Objetivos de aprendizaje

Al terminar este módulo podrás:

1. **Diseñar** la arquitectura de una plataforma de observabilidad para un sistema multi-agente a partir de **requisitos** (SLOs, privacidad, coste) y justificar cada decisión (ADR).
2. **Verificar** la instrumentación con **contratos de telemetría** automáticos (una traza por ticket, tokens solo en hojas, logs correlacionados, cero PII).
3. **Configurar** un gateway del Collector que integre redacción, métricas RED previas al muestreo, tail sampling y exportación, y **validarlo** con el binario real.
4. **Medir** el efecto del muestreo (cobertura y sesgo) y estimar el coste mensual del propio stack con supuestos explícitos.
5. **Calcular** SLIs y burn rate de una ejecución con incidente y **estimar** la calidad con intervalo de confianza.
6. **Evaluar** la plataforma con un *scorecard* automático y un checklist de producción de 31 puntos (los automáticos, medidos; los manuales, declarados con evidencia).
7. **Defender** una revisión de arquitectura: riesgos, modos de fallo, detección, mitigación, coste y plan de pruebas (incluida la mesa de incidentes).

## 🔁 Conexión con módulos anteriores

<details>
<summary>1 · (Módulo 18) ¿Por qué las métricas RED se calculan antes del tail sampling?</summary>

Porque el tail sampling conserva sobre todo errores y lentas: cualquier métrica derivada de lo que sobrevive está sesgada (2,6 % → ≈ 12 % de errores en el módulo 18). `span_metrics` en `traces/in` + `forward` calcula sobre el 100 %.

</details>

<details>
<summary>2 · (Módulo 20) ¿Qué dos capas de redacción usa SupportOps y por qué no basta una?</summary>

Redacción en la aplicación (HMAC) y en el Collector (`redaction`). Ninguna sola basta: el Collector no ve lo que no pasa por él ni puede detectar nombres; la aplicación puede olvidar un logger. Defensa en profundidad.

</details>

<details>
<summary>3 · (Módulo 16) ¿Qué alertas y por qué ventanas múltiples?</summary>

Burn rate multiventana: ventana larga (¿es grave?) y corta (¿sigue ocurriendo?). 14,4× en 1 h/5 min para páginas y 1× en 3 d/6 h para tickets; en el laboratorio, ventanas comprimidas.

</details>

<details>
<summary>4 · (Módulo 11) ¿Cómo se cuenta el coste en un árbol de agentes?</summary>

Solo en las hojas `chat`: los `invoke_agent` repiten el total de sus hijos; sumar todos los spans multiplica el coste.

</details>

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import collections, dataclasses, hashlib, json, math, re, statistics, subprocess, tempfile, time, uuid, pathlib
import yaml
from IPython.display import Markdown
from supportops import incidents as I, privacy as P, sampling as S
from supportops.multiagent import build_mesh, MultiAgentSupportOps, DelegationPolicy, cost_rollup, waterfall_services
from supportops.traffic import Incident
from supportops.pricing import cost_usd
from supportops.quality_eval import wilson_interval
STACK = check_stack("core+metrics")
LIVE = not STACK.degraded
DOCKER = P.docker_available()
PLATFORM = pathlib.Path(ROOT) / "22_capstone" / "platform"

In [ ]:
def run_platform(n: int = 480, seed: int = SEED, redact: bool = True, content: bool = True, mode: str = "memory", endpoint: str | None = None):
    """La plataforma completa de SupportOps: multi-agente (orquestador + 4 especialistas) con telemetría OTel por servicio, captura de contenido configurable,
    redacción en la app, políticas de resiliencia y una caída parcial (la herramienta `get_order` del billing-agent agota plazos entre los minutos 10 y 20 de una ejecución de 30 min). Devuelve (mesh, tickets, resultados)."""
    mesh = build_mesh(mode=mode, endpoint=endpoint)
    red = P.Redactor("hash", names=True) if redact else None
    for a in mesh.handles:
        mesh.instr(a, capture_content=content, redact=red)
    system = MultiAgentSupportOps(mesh, mode="delegate", policy=DelegationPolicy(timeout_s=6, max_retries=2, backoff_s=0.5))
    inc = Incident("billing-timeouts", "tool_timeout", 10 * 60, 20 * 60, 1.0, {"tool": "get_order", "timeout_s": 8.0, "agent": "billing-agent", "sticky": True})
    tickets = generate_tickets(n, seed, duration_s=1800, fault_rates=DEFAULT_FAULT_RATES, incidents=[inc], pii_rate=0.3)
    results = [system.run(t) for t in tickets]
    mesh.force_flush()
    return mesh, tickets, results

In [ ]:
def pii_after(mesh):
    txt = [json.dumps(s["attributes"], ensure_ascii=False) for s in mesh.span_records()] + [str(l["body"]) + json.dumps(l["attributes"], ensure_ascii=False) for l in mesh.log_records()]
    return P.pii_exposure(txt)

def scorecard(mesh, tickets, results, corpus_cov=None, cfg_ok=None, quality=None):
    """Comprobaciones automáticas de la plataforma con SU PROPIA telemetría. Devuelve una lista de filas (id del checklist, área, medida, umbral, ok)."""
    sp, lg = mesh.span_records(), mesh.log_records()
    rows = []
    def add(cid, area, what, value, thr, ok): rows.append({"id": cid, "área": area, "comprobación": what, "valor": value, "umbral": thr, "ok": bool(ok)})
    tt = I.trace_table(sp, tickets[0].created_at - 1)
    add("I1", "Instrumentación", "trazas por ticket", round(len(tt) / len(tickets), 3), "= 1", len(tt) == len(tickets))
    leaf = collections.defaultdict(int)
    for s in sp:
        if s["name"].startswith("chat"): leaf[s["trace_id"]] += int(s["attributes"].get("gen_ai.usage.input_tokens", 0) or 0)
    tree = sum(r.input_tokens for r in results)
    add("I3", "Instrumentación", "tokens de entrada: hojas chat / árbol del orquestador", round(sum(leaf.values()) / tree, 3), "= 1", sum(leaf.values()) == tree)
    cov = np.mean([bool(l["trace_id"]) for l in lg]); add("I6", "Instrumentación", "logs con trace_id", round(float(cov), 3), "= 1", cov == 1.0)
    good = np.mean([r.status in ("ok", "escalated") for r in results]); add("S1", "SLOs", "disponibilidad (objetivo 99 %: se mide, no se exige cumplirlo en el incidente)", round(float(good), 4), "medida", True)
    p95 = float(np.percentile([r.latency_s for r in results], 95)); add("S1", "SLOs", "p95 de la petición (s)", round(p95, 2), "medida", True)
    c = [r.cost_usd for r in results]; add("K1", "Coste", "coste por ticket (USD, ilustrativo) y p95/media", f"{np.mean(c):.5f} · {np.percentile(c, 95) / np.mean(c):.2f}×", "medida", True)
    ex = pii_after(mesh); no_names = sum(v for k, v in ex.items() if k != "name")
    add("P2", "Privacidad", "PII residual en spans+logs (sin nombres)", no_names, "= 0", no_names == 0)
    if quality is not None: add("Q3", "Calidad", "groundedness (muestra uniforme 25 %): límite inferior del IC 95 %", round(quality["lo"], 3), "≥ 0,85", quality["lo"] >= 0.85)
    if corpus_cov is not None:
        add("C3", "Collector", "recall de errores tras el tail sampling (%)", round(corpus_cov["recall_error_pct"], 1), "= 100", corpus_cov["recall_error_pct"] >= 99.9)
    if cfg_ok is not None: add("C2", "Collector", "configuración del gateway válida (otelcol validate)", cfg_ok, "OK", cfg_ok is True)
    return pd.DataFrame(rows)

---
# 1 · De requisitos a arquitectura

Una plataforma se justifica por las **preguntas que debe responder** y los **riesgos que debe cubrir**, no por las herramientas. Esta tabla (la de la revisión de arquitectura) conecta cada requisito con el módulo donde se construyó y la decisión tomada:

| Requisito de negocio | Medida (SLI) | Decisión de plataforma | Módulos |
|---|---|---|---|
| El agente responde | ratio de tickets `ok` o `escalated` ≥ 99 % | SLO + burn rate multiventana provisionado como código | 09, 10, 16 |
| Responde rápido | p95 de `invoke_workflow` ≤ 8 s | histogramas con buckets en el SLO; plazos propagados y disyuntor entre agentes | 09, 11 |
| Responde bien | groundedness ≥ 95 % (muestra uniforme) | evaluación online + compuerta de regresión + canario | 14, 15 |
| Cuesta lo previsto | USD por ticket resuelto; sin bucles | tokens como verdad, `BudgetGuard`/`LoopGuard`, coste solo en hojas | 11, 13 |
| No filtra datos | 0 PII en backends; inyección detectada | captura apagada, redacción en app y Collector, detector + comportamiento, auditoría | 20 |
| Es depurable | traza por ticket; versión en spans; runbooks | contratos de telemetría, mesa de incidentes | 03, 15, 21 |
| Escala sin arruinarnos | coste del stack ≤ presupuesto | tail sampling por valor, métricas antes, retención por clase | 18 |
| Se despliega con confianza | configuración validada, versiones fijadas | Helm + valores como código + `otelcol validate` en CI | 05, 19 |

In [ ]:
fig, ax = viz.canvas(13.5, 6.6, (0, 135), (0, 66))
P_ = viz.PALETTE
ax.text(1, 63, "Arquitectura de la plataforma de SupportOps", fontsize=12, fontweight="bold")
cli = viz.box(ax, 1, 28, 12, 9, "Clientes\n(tickets)", P_["neutral"], fc="#f3f4f6", fontsize=8)
orch = viz.box(ax, 18, 28, 15, 9, "Orquestador\nsupportops-\norchestrator", P_["traces"], fontsize=8, bold=True)
subs = [viz.box(ax, 40, 53.5 - i * 9.5, 15, 7.5, t, P_["traces"], fontsize=7.5) for i, t in enumerate(["orders-agent", "billing-agent", "knowledge-agent", "escalation-agent"])]
tools = viz.box(ax, 61, 26, 15, 18, "Herramientas /\nKB / MCP\n(idempotentes)", P_["profiles"], fontsize=8)
llm = viz.box(ax, 61, 49, 15, 8, "Proveedor LLM", P_["warn"], fontsize=8)
viz.arrow(ax, viz.right(cli), viz.left(orch)); [viz.arrow(ax, viz.right(orch), viz.left(b), "", P_["traces"]) for b in subs]
for b in subs[1:]: viz.arrow(ax, viz.right(b), (60.7, b[1] + 3.5), "", P_["profiles"])
viz.arrow(ax, (55.3, 57), (60.7, 52.5), "", P_["warn"])
col = viz.box(ax, 84, 24, 20, 20, "Collector gateway\n① memory_limiter\n② redaction\n③ span_metrics (RED)\n④ tail_sampling\n⑤ batch → export", P_["metrics"], fontsize=7.8, bold=True)
viz.arrow(ax, (76.3, 33), (83.7, 33), "OTLP", P_["metrics"])
be = [viz.box(ax, 112, 46, 21, 8, "Tempo · trazas", P_["traces"], fontsize=8), viz.box(ax, 112, 33, 21, 8, "Loki · logs (trace_id)", P_["logs"], fontsize=8), viz.box(ax, 112, 20, 21, 8, "Prometheus · RED/GenAI", P_["metrics"], fontsize=8)]
[viz.arrow(ax, viz.right(col), viz.left(b), "", P_["metrics"]) for b in be]
graf = viz.box(ax, 112, 4, 21, 9, "Grafana · SLOs · alertas\nrunbooks · dashboards 00-22", P_["ok"], fontsize=7.8, bold=True)
[viz.arrow(ax, viz.bottom(b) if i == 2 else (b[0] + b[2] / 2, b[1] - 0.4), (b[0] + b[2] / 2, 13.6) if i == 2 else (b[0] + b[2] / 2, b[1] - 3.2), "", P_["ok"]) for i, b in enumerate(be)]
ev = viz.box(ax, 84, 4, 20, 9, "Evaluador online\n(juez) + canario", P_["profiles"], fontsize=7.8); viz.arrow(ax, viz.top(ev), viz.bottom(col), "eventos", P_["profiles"])
sec = viz.box(ax, 2, 3, 74, 7, "Seguridad: captura apagada · HMAC en app · detector de inyección + comportamiento · auditoría", P_["error"], fc="#fdecec", fontsize=7.5)
ops = viz.box(ax, 2, 13, 74, 7, "Despliegue: Helm + valores como código · versiones fijadas · otelcol validate en CI", P_["neutral"], fc="#f3f4f6", fontsize=7.5)
plt.show()

> 👀 **Qué debes observar**
>
> - El Collector es el **punto de control único**: redacción (privacidad), métricas RED (SLO) y muestreo (coste) pasan por él **en ese orden**, y por eso el orden de los procesadores es una decisión de arquitectura, no de estilo.
> - Los agentes solo conocen un endpoint OTLP. Eso permite cambiar muestreo, redacción o destinos **sin redesplegar agentes** (módulo 00).
> - Las franjas inferiores (seguridad, despliegue) **atraviesan** todas las capas: son requisitos transversales, no componentes.

---
# 2 · Instrumentación y contratos de telemetría

Ejecutamos la plataforma (multi-agente, contenido capturado pero **redactado en la app**, una caída parcial entre los minutos 10 y 20 de una ejecución de 30 min) y la sometemos a **contratos**: afirmaciones sobre la telemetría que, si dejan de cumplirse, te avisan **antes** que un usuario.

In [ ]:
N = 480 if FAST_DEV_RUN else 960
MESH, TICKETS, RES = run_platform(N)
SP, LG = MESH.span_records(), MESH.log_records()
print(f"{len(TICKETS)} tickets · {len(SP)} spans · {len(LG)} logs · servicios: {sorted({s['resource']['service.name'] for s in SP})}")
print("estados:", dict(collections.Counter(r.status for r in RES)), "· reintentos:", sum(r.retries for r in RES), "· timeouts:", sum(r.timeouts for r in RES))
sc = scorecard(MESH, TICKETS, RES)
display(sc.set_index("id"))

In [ ]:
ok_traces = [r for r in RES if r.n_tool_calls >= 1 and r.agents_called][:1]
tid = ok_traces[0].trace_id
fig, ax = plt.subplots(figsize=(12, 5)); waterfall_services([s for s in SP if s["trace_id"] == tid], ax=ax, title=f"Un ticket de extremo a extremo ({tid[:8]}…): orquestador y especialistas en una sola traza", hide_chat=True)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar**
>
> - La tabla es el **contrato** ejecutado: una traza por ticket (`1,0`), tokens de entrada de las hojas = tokens del árbol (`1,0`: nada duplicado), logs con `trace_id` al 100 % y **cero PII** residual (sin nombres) pese a que el 30 % de los tickets trae correo y teléfono y hay fugas inyectadas.
> - Cada contrato es una línea de CI: si mañana un cambio de librería rompe la propagación, `trazas por ticket` deja de ser 1 y el pipeline lo detiene antes de producción (módulo 21, caso E).
> - La cascada de la traza muestra los **saltos entre servicios** (orquestador → especialista) con colores por servicio: es la prueba visual de que el contexto se propaga.

In [ ]:
rows = [s for s in SP]
roll = cost_rollup(rows)
naive = collections.defaultdict(float)
for s in rows:
    a = s["attributes"]; i, o = a.get("gen_ai.usage.input_tokens", 0) or 0, a.get("gen_ai.usage.output_tokens", 0) or 0
    if i or o: naive[s["resource"]["service.name"]] += cost_usd(a.get("gen_ai.response.model") or a.get("gen_ai.request.model") or "mock-small", i, o)
names = sorted(roll)
fig, ax = plt.subplots(figsize=(10, 3.8)); x = np.arange(len(names))
ax.bar(x - 0.2, [roll[n]["cost_usd"] * 1000 for n in names], 0.4, label="solo hojas chat (correcto)", color=viz.PALETTE["ok"])
ax.bar(x + 0.2, [naive.get(n, 0) * 1000 for n in names], 0.4, label="sumando todos los spans (doble conteo)", color=viz.PALETTE["error"])
ax.set_xticks(x); ax.set_xticklabels([n.replace("supportops-", "") for n in names], fontsize=8); ax.set_ylabel("coste (milésimas de USD, ilustrativo)"); ax.legend(fontsize=8); ax.set_title("Coste por agente: contarlo mal lo infla")
plt.tight_layout(); plt.show()
print(f"coste total hojas = {sum(r['cost_usd'] for r in roll.values()):.4f} USD · suma ingenua = {sum(naive.values()):.4f} USD · coste del árbol según los resultados = {sum(r.cost_usd for r in RES):.4f} USD")

> 👀 **Qué debes observar**
>
> - El **orquestador** acumula en su `invoke_agent` los tokens de sus hijos: sumar todo cuenta de más (barras rojas por encima de las verdes en el orquestador). El coste por agente correcto sale de las **hojas** y coincide con el coste del árbol de los resultados.
> - La atribución por agente permite decidir dónde optimizar (p. ej. el sintetizador `mock-large` del orquestador y el `billing-agent` concentran el gasto).

---
# 3 · El gateway del Collector: la pieza que integra todo

El archivo `platform/collector-gateway.yaml` (generado y versionado) integra lo aprendido. **Línea a línea lo importante:**

- `memory_limiter` primero (módulo 05): protege al proceso; los datos rechazados se devuelven al emisor, que reintenta.
- `redaction` **antes** de `forward`/`span_metrics` y del tail sampling (módulo 20): nada sin redactar sale de este pipeline ni alimenta métricas ni decisiones de muestreo.
- `traces/in` → `forward` + `span_metrics` (módulo 18): las métricas RED se calculan sobre el **100 %** de las trazas; `traces/sampled` aplica `tail_sampling` solo a lo que va a Tempo.
- Políticas de `tail_sampling`: errores, lentas (≥ 8 s), caras (`ottl_condition` sobre un atributo decimal, **no** `numeric_attribute`: SIGSEGV en 0.162.0) y 10 % de base.
- Exportadores `otlp_grpc`/`otlp_http` (nombres de 0.162) y métricas a Prometheus por su receptor OTLP.

In [ ]:
CFG_TEXT = (PLATFORM / "collector-gateway.yaml").read_text()
print(CFG_TEXT)
CFG = yaml.safe_load(CFG_TEXT)
if DOCKER:
    ok, msg = P.validate_collector_config(CFG_TEXT); CFG_OK = True if ok else msg; print("otelcol validate →", "OK" if ok else msg)
else:
    CFG_OK = None; print("(sin Docker: no se pudo ejecutar otelcol validate)")
pl = CFG["service"]["pipelines"]
checks = {"memory_limiter primero en cada pipeline": all(p["processors"][0] == "memory_limiter" for k, p in pl.items() if k != "traces/sampled"),
          "redaction antes de forward y span_metrics": pl["traces/in"]["processors"].index("redaction") < len(pl["traces/in"]["processors"]) and "forward" in pl["traces/in"]["exporters"],
          "span_metrics calculado antes del muestreo": "span_metrics" in pl["traces/in"]["exporters"] and "tail_sampling" in pl["traces/sampled"]["processors"],
          "logs también redactados": "redaction" in pl["logs"]["processors"], "batch al final": all(p["processors"][-1] == "batch" for k, p in pl.items() if k != "traces/in"),
          "sin numeric_attribute sobre decimales": all(p.get("type") != "numeric_attribute" for p in CFG["processors"]["tail_sampling"]["policies"])}
display(pd.Series(checks, name="¿cumple?").to_frame())

> 👀 **Qué debes observar**
>
> - El gateway pasa `otelcol validate` con el binario real (0.162.0) y cumple las seis reglas de diseño que el curso fue acumulando: **cada línea de la tabla es un incidente evitado** (OOM, PII, métricas sesgadas, logs sin redactar, SIGSEGV).
> - La comprobación estructural es la que debe vivir en **CI**: cuesta milisegundos y falla ante el típico «he añadido un procesador y lo he puesto en el sitio equivocado».

---
# 4 · Muestreo y coste: lo que se guarda y lo que cuesta

Con las políticas del gateway (aquí con el umbral de «lenta» del corpus, 5 s, porque el corpus es de agente único; el gateway usa 8 s para el multi-agente, más lento por construcción) evaluamos, sobre el corpus del módulo 18 (1 200 trazas del agente, con un incidente), qué se conserva y qué sesgo introduce el muestreo en las métricas derivadas **si** se calcularan después (por eso el gateway las calcula antes).

In [ ]:
C = S.build_corpus(1200 if FAST_DEV_RUN else 3000, SEED, incidents=[Incident("provider", "llm_error", 300, 420, 0.6, {"status": 503, "at_call": 0, "persistent": True})])
POL = [S.StatusCode(("ERROR",), name="errors"), S.Latency(5000, name="slow"), S.AttrCompare(key="supportops.cost_usd", op=">", value=0.004, name="costly"), S.Probabilistic(10, name="baseline")]
TR = S.TailSampler(POL, decision_wait_s=10, num_traces=50_000).run(C.traces)
COV = S.coverage(TR.kept, C)
W = S.inclusion_weights(TR.kept, lambda t: t.interesting, 0.10)
truth, naive, fixed = S.estimate_metrics(C.traces), S.estimate_metrics(TR.kept), S.estimate_metrics(TR.kept, W)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
keys = ["recall_error_pct", "recall_slow_pct", "recall_costly_pct", "kept_traces_pct", "kept_bytes_pct"]; labels = ["errores", "lentas", "caras", "trazas guardadas", "bytes guardados"]
axes[0].barh(labels, [COV[k] for k in keys], color=[viz.PALETTE["ok"]] * 3 + [viz.PALETTE["traces"]] * 2); axes[0].set_xlim(0, 105); axes[0].set_xlabel("%"); axes[0].set_title("Cobertura del muestreo de la plataforma"); axes[0].invert_yaxis()
vals = [truth["error_rate"] * 100, naive["error_rate"] * 100, fixed["error_rate"] * 100]
axes[1].bar(["verdad\n(100 % de las trazas)", "tras muestrear\n(sin pesos)", "tras muestrear\n(con pesos)"], vals, color=[viz.PALETTE["neutral"], viz.PALETTE["error"], viz.PALETTE["ok"]]); axes[1].set_ylabel("tasa de error (%)"); axes[1].set_title("El sesgo de calcular RED después de muestrear")
for i, v in enumerate(vals): axes[1].text(i, v + 0.2, f"{v:.1f}", ha="center")
plt.tight_layout(); plt.show()
print({k: round(COV[k], 1) for k in keys}, "· error_rate verdad/naive/pesos:", round(truth["error_rate"] * 100, 2), round(naive["error_rate"] * 100, 2), round(fixed["error_rate"] * 100, 2), "· dropped_too_early:", TR.dropped_too_early)

> 👀 **Qué debes observar**
>
> - La política conserva **el 100 % de los errores, las lentas y las caras** con una fracción pequeña del total de trazas y bytes: el muestreo **por valor** es lo que hace asequible la plataforma.
> - Pero la tasa de error calculada *después* del muestreo se infla varias veces (barra roja) frente a la verdad (gris); con pesos de inclusión se corrige (verde). El gateway evita el problema calculando las métricas RED **antes** (`traces/in`).
> - `dropped_too_early` = 0: `num_traces` es suficiente para el tráfico y la espera. Si no lo fuera, perderías trazas **sin error** (módulo 18).

In [ ]:
KEEP = COV["kept_traces_pct"] / 100
cm_full = S.StackCostModel.from_corpus(C, 1_000_000, trace_keep=1.0, log_keep=1.0, ingest_price_gb=0.5)      # escenario SaaS ILUSTRATIVO: 0,5 USD por GB ingerido
cm_tail = S.StackCostModel.from_corpus(C, 1_000_000, trace_keep=KEEP, log_keep=KEEP, ingest_price_gb=0.5)
RET = {"traces": 14, "logs": 7, "metrics": 30}
c_full, c_tail = cm_full.monthly_cost(RET), cm_tail.monthly_cost(RET)
fig, ax = plt.subplots(figsize=(8, 3.8)); x = np.arange(3); sig = ["traces", "logs", "metrics"]
ax.bar(x - 0.2, [c_full[s] for s in sig], 0.4, label="guardarlo todo", color=viz.PALETTE["error"]); ax.bar(x + 0.2, [c_tail[s] for s in sig], 0.4, label=f"tail sampling ({KEEP:.0%} de las trazas)", color=viz.PALETTE["ok"])
ax.set_xticks(x); ax.set_xticklabels(sig); ax.set_ylabel("USD/mes (ILUSTRATIVO, 1 M tickets/día)"); ax.legend(); ax.set_title("Coste mensual del propio stack")
plt.tight_layout(); plt.show()
print({k: round(v, 1) for k, v in cm_full.daily_gb().items()}, "GB/día (todo) ·", {k: round(sum(c.values()), 1) for k, c in (("todo", {"t": sum(c_full.values())}), ("tail", {"t": sum(c_tail.values())}))})

> 👀 **Qué debes observar**
>
> - El coste de **trazas y logs** baja en proporción a lo conservado; **las métricas no cambian** (su coste está en las series, no en el volumen de tráfico): otra razón para controlar la cardinalidad.
> - Estos USD son **ilustrativos** (precios configurables del módulo 18): la lección es la **proporción** y la sensibilidad (¿qué pasa si el tráfico se duplica o la retención de logs pasa a 30 días?), que debes incluir en tu revisión de arquitectura.

---
# 5 · SLOs y alertas sobre la ejecución con incidente

La caída es **parcial**: solo la ruta de reembolsos (el `billing-agent`, ≈ 20 % del tráfico) agota plazos entre los minutos 10 y 20, y el sistema **degrada bien** (reintentos, escalado a humano). Calculamos dos SLIs —**disponibilidad** (`ok` o `escalated`) y **latencia** (≤ 10 s, el objetivo medido
sobre la línea base de un multi-agente: más cadena, más latencia que el agente único)— y aplicamos burn rate multiventana de laboratorio (larga 5 min, corta 1 min). Una caída parcial rara vez quema a 14,4×: con el 20 % del tráfico afectado el máximo teórico de la fracción mala es ≈ 20 %, es decir 4× el presupuesto del 5 % de la latencia (y, en disponibilidad, solo una fracción pequeña de esos tickets acaba mal porque el sistema degrada y escala a humano, que cuenta como disponible): por eso aquí usamos la
política de **ticket (≥ 3×)** y no la de página (módulo 16: dimensiona la política al tipo de incidente).

In [ ]:
from supportops.clock import SIM_EPOCH_S as T0
MINS = 30; INC = (10, 20)
def per_minute(pred):
    good = np.zeros(MINS); tot = np.zeros(MINS)
    for r in RES:
        m = int((r.start_s - T0) // 60)
        if 0 <= m < MINS: tot[m] += 1; good[m] += pred(r)
    return good, tot
def burn(g, t, w, obj):
    cg, ct = np.cumsum(np.insert(g, 0, 0)), np.cumsum(np.insert(t, 0, 0)); i = np.arange(len(g)); lo = np.maximum(i + 1 - w, 0)
    gg, tt = cg[i + 1] - cg[lo], ct[i + 1] - ct[lo]
    with np.errstate(divide="ignore", invalid="ignore"): return np.where(tt >= 5, (1 - gg / np.maximum(tt, 1)) / (1 - obj), 0.0)
SLI = {"disponibilidad (ok|escalated)": (lambda r: r.status in ("ok", "escalated"), 0.99), "latencia ≤ 10 s": (lambda r: r.latency_s <= 10, 0.95)}
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8)); FIRES = {}
for ax, (name, (pred, obj)) in zip(axes, SLI.items()):
    g, t = per_minute(pred); bl, bs = burn(g, t, 5, obj), burn(g, t, 1, obj); fire = (bl >= 3) & (bs >= 3); FIRES[name] = fire
    ax.plot(bl, label="burn (5 min)", color=viz.PALETTE["metrics"]); ax.plot(bs, label="burn (1 min)", color=viz.PALETTE["traces"], alpha=0.5); ax.axhline(3, color="k", ls=":"); ax.axvspan(*INC, color="#d64545", alpha=0.12, label="caída real")
    ax.scatter(np.where(fire)[0], bl[fire], color="red", zorder=5, label="alerta"); ax.set_title(f"{name} · objetivo {obj:.0%}"); ax.set_xlabel("minuto"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
for name, f in FIRES.items(): print(f"{name:34} primer disparo: {int(np.argmax(f)) if f.any() else None} · falsas alarmas antes del minuto {INC[0]}: {int(f[:INC[0]].sum())} · minutos en alerta: {int(f.sum())}")

> 👀 **Qué debes observar**
>
> - Las **dos** alertas saltan **dentro** de la caída y no antes: la de disponibilidad casi de inmediato (los pocos tickets `degraded` bastan para una quema de ticket) y la de **latencia** unos minutos después (necesita acumular en la ventana larga); la mayor parte del daño está en la latencia, porque el sistema **degrada bien** (reintentos y escalado a humano cuentan como disponibles). Qué SLI es el «de verdad» es una decisión que defenderás en la revisión.
>     - Una alerta de **página** (14,4×) no saltaría con un 20 % del tráfico afectado y un sistema que degrada bien: la política (ticket, 3×) debe corresponder al tipo de fallo. Lo contrario es la receta de las alertas que nadie atiende.

In [ ]:
UNI = [r for r in RES if int(hashlib.sha256(r.ticket_id.encode()).hexdigest(), 16) % 4 == 0]          # muestra uniforme determinista del 25 %
k, n = sum(r.grounded for r in UNI), len(UNI); lo, hi = wilson_interval(k, n)
QUALITY = {"p": k / n, "lo": lo, "hi": hi, "n": n}
print(f"groundedness (muestra uniforme 25 %): {k}/{n} = {k / n:.3f} · IC 95 % Wilson [{lo:.3f}, {hi:.3f}] · verdad de toda la población: {np.mean([r.grounded for r in RES]):.3f} · objetivo 95 %")

---
# 6 · Privacidad: antes y después

Misma ejecución con y sin redacción en la aplicación (captura de contenido encendida en ambas), y lo que quedaría si además pasara por el `redaction` del gateway (medido en el módulo 20).

In [ ]:
M_RAW, T_RAW, R_RAW = run_platform(N, redact=False, content=True)
M_OFF, T_OFF, R_OFF = run_platform(N, redact=False, content=False)
ex = {"captura apagada (por defecto)": pii_after(M_OFF), "captura + SIN redacción": pii_after(M_RAW), "captura + redacción en la app": pii_after(MESH)}
fig, ax = plt.subplots(figsize=(9, 3.8)); bot = np.zeros(3); names = list(ex)
for kind, col in [("email", viz.PALETTE["error"]), ("phone", viz.PALETTE["warn"]), ("card", viz.PALETTE["profiles"]), ("name", viz.PALETTE["traces"])]:
    v = np.array([ex[n].get(kind, 0) for n in names], float); ax.bar(range(3), v, bottom=bot, label=kind, color=col); bot += v
ax.set_xticks(range(3)); ax.set_xticklabels(names, fontsize=8); ax.set_ylabel("valores de PII en spans + logs"); ax.legend(); ax.set_title("Exposición de PII en la plataforma multi-agente")
plt.tight_layout(); plt.show()
print({k: sum(v.values()) for k, v in ex.items()})

> 👀 **Qué debes observar**
>
> - Con la captura apagada hay **cero** PII, con captura sin redactar hay cientos de valores repartidos por los **cinco servicios** (cada salto del multi-agente vuelve a copiar el texto) y con redacción en la app vuelve a cero.
> - En un multi-agente la PII **se multiplica por el número de saltos** (el orquestador pasa el texto a cada especialista). Por eso la redacción va en **cada** servicio (o en el gateway) y no solo en el primero.

---
# 7 · Despliegue: lo verificable

El módulo 19 desplegó los componentes con Helm. Aquí **comprobamos** lo que el capstone puede afirmar con honestidad: charts y versiones fijados, valores como código presentes y el gateway del capstone coherente con los valores de Kubernetes.

In [ ]:
from supportops import k8s as K
rows = [{"release": k, "chart": c.chart, "versión del chart": c.version, "app": c.app_version, "propio": c.own} for k, c in K.CHARTS.items()]
display(pd.DataFrame(rows).set_index("release"))
vals = sorted(p.name for p in (pathlib.Path(ROOT) / "19_kubernetes_helm" / "k8s" / "values").glob("*.yaml"))
print("valores de Helm como código:", vals)
print("imágenes sin 'latest' en el stack:", all("latest" not in l for l in (pathlib.Path(ROOT) / "stack" / "docker-compose.yml").read_text().splitlines() if "image:" in l))

> **⚠️ Cuidado · Qué NO se verificó en el despliegue**
>
> En el entorno de construcción del curso (sandbox sin red abierta y con *runc* anidado) **ningún pod llegó a ejecutarse** en un clúster: se validaron renderizado, esquemas, configuraciones con los binarios y el flujo de telemetría con laboratorios en Docker (módulo 19).
> Un despliegue real exige además: *probes*, PDB y HPA bajo carga, afinidad por `traceID` con ≥ 2 réplicas del gateway, redes y secretos reales. Márcalo en tu checklist como **manual**.

---
# 8 · Scorecard integral y checklist de producción

Reunimos todo en un **scorecard** automático (las comprobaciones que la propia telemetría permite) y lo cruzamos con el **checklist de producción** (`platform/production-checklist.yaml`, 31 puntos, 15 automáticos). Los manuales se declaran con evidencia: aquí
marcamos cuáles cubre el curso con artefactos entregados (dashboards, runbooks, valores de Helm, plantillas) y cuáles quedan **pendientes** de tu organización.

In [ ]:
SC = scorecard(MESH, TICKETS, RES, corpus_cov=COV, cfg_ok=CFG_OK, quality=QUALITY)
display(SC.set_index("id"))
CHK = yaml.safe_load((PLATFORM / "production-checklist.yaml").read_text())["items"]
auto_ok = set(SC[SC["ok"]]["id"]); auto_all = set(SC["id"])
# manuales con evidencia ENTREGADA por el curso (artefactos en el repositorio); el resto: pendiente en TU organización
EVID = {"I2": "instantánea de la spec (04)", "I4": "VersionTagInstrumentor (15)", "I5": "ParentBased (11)", "I7": "max_retries=0 (06)", "C4": "dropped_too_early (18)", "C5": "load_balancing (19)",
        "S3": "runbooks del módulo 16", "Q1": "OnlineEvaluator (14)", "Q2": "ReleaseGate (15)", "K2": "BudgetGuard/LoopGuard (13)", "P3": "InjectionDetector/BehaviorMonitor (20)", "P4": "AuditTrail (20)",
        "D2": "dashboards y alertas aprovisionados (10, 16)", "D3": "límites por tenant (18)"}
PENDING = {"S4": "política pactada con producto", "R1": "mesa de incidentes superada (tú)", "R2": "contratos en CI y game days (tu organización)"}
status = []
for it in CHK:
    cid = it["id"]
    if it["automatica"]: st = "medido" if cid in auto_ok else ("falla" if cid in auto_all else "sin medir aquí")
    else: st = "evidencia entregada" if cid in EVID else "pendiente (tuyo)"
    status.append({**it, "estado": st})
CK = pd.DataFrame(status)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
areas = list(dict.fromkeys(CK["area"])); order = ["medido", "evidencia entregada", "sin medir aquí", "pendiente (tuyo)", "falla"]; colors = [viz.PALETTE["ok"], viz.PALETTE["traces"], viz.PALETTE["warn"], viz.PALETTE["neutral"], viz.PALETTE["error"]]
bot = np.zeros(len(areas))
for st, col in zip(order, colors):
    v = np.array([((CK["area"] == a) & (CK["estado"] == st)).sum() for a in areas], float)
    if v.sum(): axes[0].barh(areas, v, left=bot, label=st, color=col); bot += v
axes[0].invert_yaxis(); axes[0].legend(fontsize=7, loc="lower right"); axes[0].set_title("Checklist de producción por área (31 puntos)")
ca = SC.groupby("área")["ok"].mean(); ang = np.linspace(0, 2 * np.pi, len(ca), endpoint=False); axes[1].remove(); ax2 = fig.add_subplot(122, polar=True)
ax2.plot(np.append(ang, ang[0]), np.append(ca.values, ca.values[0]), color=viz.PALETTE["traces"], lw=2); ax2.fill(ang, ca.values, alpha=0.2, color=viz.PALETTE["traces"]); ax2.set_xticks(ang); ax2.set_xticklabels(ca.index, fontsize=8); ax2.set_ylim(0, 1); ax2.set_title("Scorecard automático por área", pad=18)
plt.tight_layout(); plt.show()
print(CK["estado"].value_counts().to_dict())

> 👀 **Qué debes observar**
>
> - Todo lo **automático** está en verde: la plataforma cumple sus propios contratos en la ejecución de referencia. Eso **no** significa «lista para producción»: de los 31 puntos, los **manuales** son los que de verdad deciden (política de presupuesto pactada, game days, contratos en CI, mesa superada).
> - El valor del checklist es separar lo que **sabes** (medido), lo que **tienes** (artefactos), lo que **no has verificado aquí** (despliegue real) y lo que **te toca** (cultura y proceso). Esa honestidad es la que se defiende en la revisión de arquitectura.

---
# 9 · Revisión de arquitectura

La entrega final es una **revisión de arquitectura** (`platform/architecture-review-template.md`): un documento corto que otro ingeniero pueda **atacar**. Estructura: contexto y objetivos, requisitos medibles, arquitectura, **decisiones con alternativas descartadas**, **riesgos con detección y mitigación**, coste con sensibilidad, plan de despliegue y pruebas, y preguntas abiertas.
Una **tabla de riesgos** buena enlaza cada riesgo con la **señal** que lo detecta (de este mismo curso):

In [ ]:
RISKS = pd.DataFrame([
    ("Pérdida silenciosa de telemetría", "media", "alto", "otelcol_exporter_send_failed_*; `dropped_too_early`; traza canario e2e", "Colas con reintento; alertas sobre el pipeline", "05, 18"),
    ("Tormenta de cardinalidad", "media", "alto", "series activas ×N con tráfico plano; valores distintos por etiqueta", "Views del SDK / aggregate_labels; lint de series en CI", "09, 21"),
    ("Trazas partidas", "media", "medio", "trazas por ticket ≠ 1; raíces de subagentes", "Contrato n_traces == n_tickets; ParentBased", "11, 21"),
    ("Fuga de PII", "media", "muy alto", "detector en la salida; contador de PII por tipo; consulta LogQL de control", "Captura apagada; redacción en app y Collector; retención corta", "20"),
    ("Regresión silenciosa de prompt", "alta", "alto", "herramientas por ticket por versión; groundedness; 👎", "Compuerta apareada + canario SPRT + rollback", "15, 21"),
    ("Cascada de reintentos", "media", "alto", "intentos por ticket; timeouts por servicio", "Disyuntor, plazos propagados, degradación", "11, 21"),
    ("Coste desbocado (bucle)", "media", "alto", "coste por ticket p99; trazas con ≥ 5 herramientas idénticas", "LoopGuard, BudgetGuard, max_steps", "13, 21"),
    ("Inyección de prompts", "alta", "alto", "puntuación de inyección; refund sin consulta previa", "Mínimo privilegio, aprobación humana, aislar contenido no fiable", "20"),
], columns=["riesgo", "probabilidad", "impacto", "señal que lo detecta", "mitigación", "módulos"])
display(RISKS)

> 👀 **Qué debes observar**
>
> - Cada riesgo tiene **una señal concreta** y **una mitigación**; ninguno depende de «estar atento». Si no puedes nombrar la señal que lo detecta, no tienes observabilidad de ese riesgo.
> - Los riesgos más probables (regresión silenciosa, inyección) son justo los que **no** disparan alertas clásicas: son los que decide la mesa de incidentes (módulo 21).

---
# 10 · Plan de pruebas: la mesa de incidentes como examen de aceptación

La plataforma se acepta cuando un ingeniero que no la construyó resuelve los **cinco incidentes de la mesa** (módulo 21) en menos de 15 minutos **con solo telemetría**. Medimos aquí una versión automática: el *tiempo de cómputo* del diagnóstico de referencia y la puntuación de `I.grade`.

In [ ]:
t0 = time.perf_counter(); b = I.build_incident("INC-E"); f = I.reference_facts("INC-E", b)
M = pd.DataFrame(b.metrics); wa, wb, _ = (lambda x: (int(np.argmax(x > 1.5)), 0, 0))(M["traces_per_ticket"].values)
diag = {"cause": "context_propagation_broken", "start_min": wa, "end_min": 59, "culprit": None, "evidence": {"orphan_share_after": f["orphan_share_after"]}, "mitigation": "restore_propagation"}
g = I.grade("INC-E", diag, b); print(f"INC-E · puntuación {g['score']}/{g['of']} · {time.perf_counter() - t0:.2f} s de cómputo · comprobaciones:", g["checks"])

---
# 11 · Con el stack: el gateway del capstone contra Tempo, Loki y Prometheus reales

Levantamos el **gateway del capstone** (`collector-gateway.yaml`) como contenedor efímero **en la red del stack** (`agentobs_default`, resuelve `tempo`, `loki` y `prometheus`) y le enviamos la telemetría del multi-agente por OTLP/HTTP.
Comprobamos tres cosas con servicios reales: que Tempo recibe **solo** lo que el muestreo decide (errores y lentas casi completos, base ≈ 10 %), que Loki recibe logs **ya redactados** y que Prometheus recibe métricas **RED de todas las trazas** (span metrics) aunque Tempo no las vea.

In [ ]:
class GatewayLab:
    """Collector del capstone en la red del stack (puerto OTLP/HTTP aleatorio en el host; no choca con el Collector del stack)."""
    def __init__(self, cfg_path, network="agentobs_default"):
        self.cfg, self.net, self.name = cfg_path, network, f"capstone-gw-{uuid.uuid4().hex[:6]}"
    def __enter__(self):
        r = subprocess.run(["docker", "run", "-d", "--name", self.name, "--network", self.net, "-p", "127.0.0.1::4318", "-v", f"{self.cfg}:/etc/otelcol-contrib/config.yaml:ro", P.COLLECTOR_IMAGE, "--config=/etc/otelcol-contrib/config.yaml"], capture_output=True, text=True)
        if r.returncode: raise RuntimeError(r.stderr[:400])
        self.port = int(subprocess.run(["docker", "port", self.name, "4318/tcp"], capture_output=True, text=True).stdout.split(":")[-1])
        for _ in range(60):
            if "Everything is ready" in self.logs(): break
            time.sleep(0.5)
        return self
    def logs(self): r = subprocess.run(["docker", "logs", self.name], capture_output=True, text=True); return r.stdout + r.stderr
    def __exit__(self, *a): subprocess.run(["docker", "rm", "-f", self.name], capture_output=True)
LIVE_RUN = None
if LIVE and DOCKER:
    net = subprocess.run(["docker", "network", "ls", "--format", "{{.Name}}"], capture_output=True, text=True).stdout.split()
    net = next((n for n in net if n.endswith("_default") and "agentobs" in n), "agentobs_default")
    with GatewayLab(str(PLATFORM / "collector-gateway.yaml"), net) as gw:
        mesh_l = build_mesh(mode="otlp", endpoint=f"http://127.0.0.1:{gw.port}")
        for a in mesh_l.handles: mesh_l.instr(a, capture_content=True, redact=None)          # SIN redacción en la app: el Collector es la red de seguridad
        system = MultiAgentSupportOps(mesh_l, mode="delegate", policy=DelegationPolicy(timeout_s=6, max_retries=1, breaker_threshold=3, propagate_deadline=True, cancel_in_flight=True))
        inc = Incident("provider", "llm_error", 100, 160, 0.8, {"status": 503, "at_call": 0, "persistent": True})
        lt = generate_tickets(150, SEED + 3, start_s="live", duration_s=240, fault_rates=DEFAULT_FAULT_RATES, incidents=[inc], pii_rate=0.4)
        lres = [system.run(t) for t in lt]; mesh_l.force_flush(); mesh_l.shutdown()
        time.sleep(30)                                        # decision_wait del tail sampling (10 s) + metrics_flush_interval de span_metrics (10 s) + exportación: no pares el gateway antes
        print(gw.logs()[-300:].replace("\n", " | "))
    LIVE_RUN = (lt, lres)
    print("Enviados", len(lt), "tickets; errores:", sum(r.status in ("error", "degraded") for r in lres), "· lentos (≥ 8 s):", sum(r.latency_s >= 8 for r in lres))
else:
    print("Sin stack o sin Docker: sección omitida (modo degradado). Comando:", STACK.command)

In [ ]:
if LIVE and LIVE_RUN:
    lt, lres = LIVE_RUN
    def poll(fn, ok, tries=45, wait=2.0):
        v = None
        for _ in range(tries):
            try: v = fn()
            except Exception: v = None
            if v is not None and ok(v): return v
            time.sleep(wait)
        return v
    def stable(fn, checks=3, wait=6.0, tries=25):
        """Espera a que el recuento deje de crecer (Tempo tarda ≈ 15-30 s en hacer buscables las trazas tras el decision_wait)."""
        last, same = -1, 0
        for _ in range(tries):
            try: n = len(fn())
            except Exception: n = -1
            same = same + 1 if (n == last and n > 0) else 0
            last = n
            if same >= checks: break
            time.sleep(wait)
        return last
    n_tempo = stable(lambda: backends.tempo_search('{ name =~ "invoke_workflow.*" }', 10, 500).get("traces", []))
    SVCS = "supportops-(orchestrator|orders-agent|billing-agent|knowledge-agent|escalation-agent)"      # solo los servicios de ESTA ejecución (no mezcles trazas/logs de otras lecciones recientes)
    slow_t = backends.tempo_search('{ name =~ "invoke_workflow.*" && trace:duration > 8s }', 10, 500).get("traces", [])
    err_t = backends.tempo_search('{ status = error && resource.service.name =~ "%s" }' % SVCS, 10, 500).get("traces", [])
    n_slow = sum(r.latency_s > 8 for r in lres)
    print(f"Tempo · trazas de workflow guardadas: {n_tempo} de {len(lt)} enviadas ({n_tempo / len(lt):.0%}) · lentas (> 8 s) guardadas: {len(slow_t)} de {n_slow} enviadas · trazas con algún span en error: {len(err_t)}")
    leak = poll(lambda: backends.loki_query('{service_name=~"%s"} |~ "[A-Za-z0-9._%%+-]+@example\\\\.com"' % SVCS, 10, 200), lambda r: True, tries=3)
    n_logs = backends.loki_query('{service_name=~"%s"}' % SVCS, 10, 500)
    print(f"Loki · líneas recibidas: {len(n_logs)} · líneas con un correo en claro: {len(leak or [])} (esperado 0: el gateway redactó)")
    red_calls = poll(lambda: backends.prom_query('sum(traces_span_metrics_calls_total{span_name="invoke_workflow support_ticket"})'), lambda r: len(r) > 0 and float(r[0]["value"][1]) >= 0.9 * len(lt), tries=45)
    print("Prometheus · span metrics (calls de invoke_workflow, TODAS las trazas):", [m["value"][1] for m in red_calls or []], "· tickets enviados:", len(lt))

> 👀 **Qué debes observar**
>
> - **Tempo** guarda una **fracción** de las trazas (entre un tercio y la mitad en esta ejecución: todas las lentas, las caras y las que tienen algún span en error —durante la caída casi todas—, más el 10 % de base), pero **Prometheus** cuenta **todas** las llamadas a `invoke_workflow` en `traces_span_metrics_calls_total`: las métricas RED salen de `traces/in` (antes del muestreo) y no sufren sesgo.
> - **Loki** recibe los logs **sin correos en claro** aunque la aplicación enviara el contenido sin redactar: el `redaction` del gateway es la red de seguridad (con sus límites: nombres y formatos ofuscados, módulo 20).
> - Si en tu ejecución los números de Tempo difieren, recuerda que tras el *decision wait* hay un retraso de indexación de ≈ 15-20 s (módulo 07): la consulta *polling* lo absorbe.

Abre el dashboard **22 · Plataforma de agentes de SupportOps** (uid `plataforma-agentes`): SLOs, calidad, coste, privacidad y la **salud del propio pipeline** (spans aceptados/enviados, `dropped_too_early`).

---
# 12 · 🏭 En producción: cómo se evalúan estas plataformas

- **Revisiones de arquitectura (*design reviews*)**: los equipos de plataforma de las grandes empresas exigen un documento con requisitos medibles, alternativas descartadas, riesgos con detección y plan de pruebas; es el formato de los ADR (*Architecture Decision Records*, Michael Nygard, 2011) y de las *production readiness reviews* de SRE (*Site Reliability Engineering*, cap. 32 «The Evolving SRE Engagement Model»).
- **Checklists de preparación para producción** (*Production Readiness*) en la práctica SRE de Google: monitorización, capacidad, respuesta a incidentes y gestión del cambio se revisan **antes** de aceptar un servicio en guardia.
- **Grafana Labs** promueve OpenTelemetry (convenciones GenAI) con Tempo, Loki y Prometheus/Mimir como base de la observabilidad de aplicaciones de IA; es el patrón que este capstone reproduce con el stack del curso. *(Afirmación general sin cita concreta: contrasta con la documentación actual de Grafana antes de citarla en tu revisión.)*
- **El coste de observabilidad** es un tema de dirección en las empresas con mucho tráfico: la combinación de *tail sampling*, métricas previas al muestreo y retención por clase es la combinación que este curso mide y recomienda para contenerlo (módulo 18); no afirmamos que sea «la forma estándar» de ninguna empresa concreta.

---
# 13 · 🧠 Secretos de la élite de la integración

1. **El orden de los procesadores es arquitectura.** `memory_limiter` → `redaction` → (`forward` → `span_metrics`) → `tail_sampling` → `batch`: invertir dos de ellos produce un incidente distinto cada vez (OOM, PII en métricas, RED sesgada, colas sin sentido).
2. **Las métricas RED nacen antes del muestreo; las decisiones de muestreo, después de la redacción.** Así ni el muestreo ni la privacidad contaminan los SLOs.
3. **Un contrato de telemetría vale más que diez dashboards.** «Una traza por ticket», «hojas = árbol», «cero PII» son tres líneas que previenen incidentes enteros.
4. **En multi-agente, la PII se multiplica por los saltos.** Redacta en cada servicio (o en el gateway), no solo en el primero.
5. **Los automáticos no sustituyen a los manuales.** El scorecard verde dice que la telemetría cumple sus contratos, no que tu organización responda a un incidente: política de presupuesto, game days y mesa superada deciden.
6. **Declara lo que NO verificaste.** Un capstone honesto separa *medido*, *entregado*, *no verificado aquí* y *pendiente*: un revisor desconfía de los verdes sin matices.
7. **El coste de la plataforma también es un SLO.** Series, bytes y retención por clase con presupuesto y alerta de desvío, igual que el coste del agente.
8. **La plataforma se acepta con un examen de incidentes**, no con una demo de dashboards: si alguien ajeno no resuelve la mesa en 15 minutos, la observabilidad no es suficiente aunque los paneles sean bonitos.
9. **Cada decisión de telemetría merece un ADR con sus consecuencias medibles.** Nygard propone un formato corto (título, estado, contexto, decisión, consecuencias) para que las decisiones de arquitectura queden como historia que se puede revisar. En observabilidad la sección «Consecuencias» debe llevar números: series nuevas, bytes/s y días de retención, qué señal se pierde con el muestreo elegido y quién paga. Un ADR de «muestreo por cola al 10 % de la base» sin la tasa de error resultante en las métricas RED es una opinión (módulo 18). *Fuente: M. Nygard, «Documenting Architecture Decisions» (15-nov-2011, Cognitect).*
10. **Haz de la preparación para producción una puerta, no un trámite.** Las *production readiness reviews* de Google SRE revisan monitorización, capacidad, respuesta a incidentes y gestión del cambio antes de aceptar un servicio en guardia; añade a esa puerta el **contrato de telemetría** (una traza por ticket, cero PII, SLO con alerta multiventana, runbook con dueño) y no aceptes en guardia lo que no lo cumpla. *Fuente: Google SRE Book, cap. 32 «The Evolving SRE Engagement Model» (título conocido del libro; no abierto desde el sandbox).*

---
# 14 · ⚠️ Errores comunes en un capstone

- Entregar una arquitectura sin requisitos medibles: nadie puede decir si funciona.
- Un scorecard que solo mide lo fácil (dashboards creados) y no los contratos (una traza por ticket, PII residual).
- Olvidar el pipeline como objeto de observación: si el Collector pierde datos, **todo lo demás** miente.
- Costes sin sensibilidad (¿y si el tráfico se duplica?) y sin supuestos explícitos.
- Dar por «hecho» el despliegue sin decir qué se probó en un clúster y qué no.
- Un solo nivel de privacidad (solo app o solo Collector) o contenido capturado «por si acaso».

---
# 15 · Autoevaluación

<details>
<summary>1 · (Cálculo) Un gateway recibe 150 trazas nuevas/s y espera 10 s antes de decidir. ¿Qué `num_traces` mínimo necesita y qué pasa si pones 1 000?</summary>

`num_traces ≥ tráfico × decision_wait = 150 × 10 = 1 500` trazas **simultáneamente** en memoria (con margen: 2-3×). Con 1 000, el Collector descarta las más antiguas **sin evaluarlas** (`sampling_trace_dropped_too_early`) y pierdes trazas —también errores— sin ningún error visible (módulo 18).

</details>

<details>
<summary>2 · (Diagnóstico) Tempo guarda el 12 % de las trazas pero el panel de «tasa de error» muestra el 14 % cuando tu SLO real es 2,5 %. ¿Qué ocurre y cómo se arregla?</summary>

Estás calculando RED con spans **ya muestreados** (el tail sampling conserva errores y lentas): sesgo al alza. Arreglo: calcular `span_metrics` en `traces/in` antes del muestreo (gateway del capstone) o corregir con pesos de inclusión.

</details>

<details>
<summary>3 · (Transferencia) Añaden un tercer nivel de agentes (subagente que delega a otro). ¿Qué cambios mínimos exige la plataforma y qué riesgos aparecen?</summary>

Instrumentación: propagación de contexto y `ParentBased` en el nuevo servicio; contratos (`n_traces == n_tickets`, hojas = árbol). SLOs: el plazo debe decrecer con la profundidad y la amplificación crece `(r+1)^d`. Privacidad: redacción en el servicio nuevo (la PII se copia otra vez). Coste/volumen: más spans por traza (revisar `num_traces` y el coste). Riesgo nuevo: cascada de reintentos de profundidad 3.

</details>

<details>
<summary>4 · Explica por qué el scorecard automático no basta para declarar la plataforma «lista para producción».</summary>

Porque mide contratos de telemetría sobre una ejecución, no la preparación de la organización: política de presupuesto pactada, runbooks usados, game days, despliegue real en un clúster con carga, límites por tenant. El checklist separa medido / entregado / no verificado / pendiente precisamente para evitarlo.

</details>

<details>
<summary>5 · (Cálculo) Si el 10 % de las trazas base se conserva y el 100 % de las interesantes, ¿qué peso tiene cada traza al corregir la tasa de error?</summary>

Interesantes: peso 1 (probabilidad de inclusión 1). Base: peso `1/0,10 = 10`. La tasa corregida es `Σ pesos de trazas con error / Σ pesos totales` (Horvitz-Thompson), que vuelve a la verdad en esperanza.

</details>

<details>
<summary>6 · (Diagnóstico) Tras desplegar la plataforma, Loki muestra correos en claro aunque el gateway tiene `redaction`. Enumera tres causas.</summary>

(a) El pipeline de **logs** no incluye el procesador; (b) hay logs que llegan a Loki por otra ruta (envío directo, otro agente); (c) el patrón no cubre el formato (ofuscado) o el dato está en un formato que la regex no ve; (d) la redacción está después de `batch`/exportador o el procesador no se activó en `service.pipelines`.

</details>

<details>
<summary>7 · (Transferencia) Defiende en 5 líneas por qué la mesa de incidentes es mejor criterio de aceptación que una demo.</summary>

Una demo muestra el camino feliz; la mesa obliga a **diagnosticar fallos desconocidos** con solo telemetría y mide tiempo y exactitud. Prueba la calidad de los datos (¿están la versión, los intentos, las herramientas?), la capacidad del equipo y la utilidad de los dashboards. Es el único examen que falla si falta una señal que importa.

</details>

---
# 16 · Referencias

- Material del curso: módulos 00-21 y sus referencias (OpenTelemetry, Collector contrib 0.162.0, Tempo 3.1, Loki 3.7, Prometheus 3.15, Grafana 13, Helm).
- M. Nygard (2011), *Documenting Architecture Decisions* (formato ADR): <https://cognitect.com/blog/2011/11/15/documenting-architecture-decisions>.
- B. Beyer, C. Jones, J. Petoff, N. R. Murphy (2016), *Site Reliability Engineering*, O'Reilly, cap. 32 «The Evolving SRE Engagement Model» (production readiness reviews).
- B. Beyer et al. (2018), *The Site Reliability Workbook*, cap. 5 «Alerting on SLOs».
- C. Majors, L. Fong-Jones, G. Miranker (2022), *Observability Engineering*, O'Reilly.
- README de `processor/redactionprocessor` y `processor/tailsamplingprocessor` y del conector `spanmetricsconnector` de `opentelemetry-collector-contrib`, v0.162.0.